# 1. Creating table

## 1.1 Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.dim_dam(
    dam_key BIGINT NOT NULL,
    dam_name STRING NOT NULL,
    height_m INT NOT NULL,
    length_m INT NOT NULL,
    capacity_Ml INT NOT NULL,
    water_supply BOOLEAN NOT NULL,
    hydro_power BOOLEAN NOT NULL,
    is_active BOOLEAN NOT NULL,

    CONSTRAINT pk_dam_key PRIMARY KEY(dam_key) RELY
)



## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.dim_dam

WITH raw_dam_names AS (
    -- distinct dams from silver table
    SELECT DISTINCT 
        TRIM(LOWER(dam_name)) AS raw_dam_name
    FROM cpt_utility_catalog.silver.silver_dam_levels_cleaned
    WHERE dam_name IS NOT NULL 
      AND LOWER(TRIM(dam_name)) != 'total_stored_big_6'
),

dam_metadata AS (
    -- reference table holding physical and operational metadata
    SELECT * FROM (VALUES
        ('theewaterskloof',  35,  640, 480188, TRUE,  FALSE, TRUE),
        ('berg_river',       68,  990, 130010, TRUE,  TRUE,  TRUE),
        ('wemmershoek',      55,  518,  58644, TRUE,  FALSE, TRUE),
        ('voelvlei',         10, 3600, 164095, TRUE,  FALSE, TRUE),
        ('steenbras_lower',  28,  420,  33517, TRUE,  TRUE,  TRUE),
        ('steenbras_upper',  31,  430,  31767, TRUE,  TRUE,  TRUE),
        ('woodhead',         23,  252,    954, FALSE, FALSE, TRUE),
        ('hely_hutchinson',  16,  528,    925, FALSE, FALSE, TRUE),
        ('de_villiers',      14,  183,    243, FALSE, FALSE, TRUE),
        ('kleinplaats',      19,  215,   1368, FALSE, FALSE, TRUE),
        ('lewis_gay',        13,  145,    182, FALSE, FALSE, TRUE),
        ('alexandra',        12,  120,    128, FALSE, FALSE, TRUE),
        ('victoria',         12,  135,    128, FALSE, FALSE, TRUE),
        ('land_en_zeezicht', 12,  200,    451, FALSE, FALSE, TRUE)
    ) AS t(raw_dam_name, height_m, length_m, capacity_Ml, water_supply, hydro_power, is_active)
),

mapped_dams AS (
    SELECT 
        xxhash64(LOWER(TRIM(s.raw_dam_name))) AS dam_key,
        INITCAP(REPLACE(s.raw_dam_name, '_', ' ')) AS dam_name,
        COALESCE(m.height_m, -1) AS height_m,
        COALESCE(m.length_m, -1) AS length_m,
        COALESCE(m.capacity_Ml, -1) AS capacity_Ml,
        COALESCE(m.water_supply, FALSE) AS water_supply,
        COALESCE(m.hydro_power, FALSE) AS hydro_power,
        COALESCE(m.is_active, TRUE) AS is_active
    FROM raw_dam_names s
    LEFT JOIN dam_metadata m ON s.raw_dam_name = m.raw_dam_name
)

-- Select standard rows combined with UNMAPPED fallback record
SELECT * FROM mapped_dams

UNION ALL

SELECT 
    xxhash64('unmapped') AS dam_key,
    'Unmapped'       AS dam_name,
    -1                   AS height_m,
    -1                   AS length_m,
    -1                   AS capacity_Ml,
    FALSE                AS water_supply,
    FALSE                AS hydro_power,
    TRUE                AS is_active;